# Lesson 5. Running the program outside the notebook

Jupyter is useful for short experiments and graphs. A controller that must
wake up on a fixed clock belongs in ordinary project files, started from a
terminal.

## Learning objectives

By the end of this lesson you will be able to:

1. Name the file that starts the simulator and the file that stores your gains.
2. Save outer PID gains to `configs/tutorial.yaml` and read them back.
3. Start the same simulator with `python run_sim.py` from a terminal.

## Glossary

| Term | Meaning |
| --- | --- |
| Module | A Python file with one job, imported by other files. The `sim` folder is a package of modules. |
| Configuration file | A text file of named numbers. YAML is the format used here. |
| Entry point | `run_sim.py`, the program you start. |
| Hidden state | A leftover variable from an earlier cell that still affects a later cell. |
| Headless | A run with no 3D window. Useful for a short report. |
| Virtual environment | A private Python install for this project, named `.venv`. |


## Step 1. Start the lesson

Run the next cell. Continue when you see `project_root:`.


In [1]:
import os
import sys

def choose_mujoco_gl(platform, display, existing):
    if existing:
        return existing
    if str(platform).startswith("linux") and not display:
        return "egl"
    return None

chosen = choose_mujoco_gl(sys.platform, os.environ.get("DISPLAY"), os.environ.get("MUJOCO_GL"))
if chosen and "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = chosen

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace

from sim.config import load_config
from sim.loop import make_sim, run_headless, summarize
from sim.pid_controller import RollSteerPID, apply_speed_refs

np.set_printoptions(precision=4, suppress=True)
print("project_root:", project_root)
print("platform:", sys.platform, "MUJOCO_GL:", os.environ.get("MUJOCO_GL", "(unset)"))


project_root: /home/lmbike2/bike-workshop/bike_mujoco_sim
platform: linux MUJOCO_GL: egl


## Step 2. Locate the project files

Each row is one file in `bike_mujoco_sim`. Run the next cell. Every line
should begin with `OK`.

| File | Role |
| --- | --- |
| `run_sim.py` | Entry point. This is the program you start. |
| `configs/tutorial.yaml` | Your gains and speeds. Text, not code. |
| `configs/new_bike.yaml` | A known-good copy. Leave it unchanged while you experiment. |
| `sim/pid_controller.py` | Outer PID: errors become a desired bar speed. |
| `sim/loop.py` | Fifty times per second: read sensors, run the PID, write motors, step physics. |
| `sim/viewer.py` | The 3D window and the keyboard. |
| `sim/joystick.py` | The on-screen stick and gamepad reading. |
| `sim/models.py` | Finds the bicycle model and checks that motors and sensors exist. |
| `models/new_bike/new_bike.xml` | MuJoCo's description of masses, wheels, and joints. |


In [2]:
paths = [
    os.path.join(project_root, "run_sim.py"),
    os.path.join(project_root, "configs", "tutorial.yaml"),
    os.path.join(project_root, "configs", "new_bike.yaml"),
    os.path.join(project_root, "sim", "pid_controller.py"),
    os.path.join(project_root, "sim", "loop.py"),
    os.path.join(project_root, "sim", "viewer.py"),
    os.path.join(project_root, "sim", "joystick.py"),
    os.path.join(project_root, "sim", "models.py"),
    os.path.join(project_root, "models", "new_bike", "new_bike.xml"),
]
for path in paths:
    rel = os.path.relpath(path, project_root)
    print(("OK " if os.path.isfile(path) else "MISSING "), rel)


OK  run_sim.py
OK  configs/tutorial.yaml
OK  configs/new_bike.yaml
OK  sim/pid_controller.py
OK  sim/loop.py
OK  sim/viewer.py
OK  sim/joystick.py
OK  sim/models.py
OK  models/new_bike/new_bike.xml


## Step 3. Save the gains, then read them back

A number stored only in a notebook variable disappears when the kernel
stops. A saved file does not.

Open `configs/tutorial.yaml` and set:

```yaml
pid_kp_roll: 7.535559389256096
pid_ki_roll: 1.2614702013440495
pid_kd_roll: 11.260517241323017
pid_kp_steer: 1.1164231977942145
```

Save the file. Then run the next cell. It reads the file from disk.

Expected result: the four printed gains match the block above. If the print
still shows 5, 0, and 6, the file was not saved. Edit it and run the cell
again.


In [3]:
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
print("File:", os.path.relpath(tutorial_yaml, project_root))
print(
    "pid_kp_roll={}\npid_ki_roll={}\npid_kd_roll={}\npid_kp_steer={}".format(
        cfg.pid_kp_roll, cfg.pid_ki_roll, cfg.pid_kd_roll, cfg.pid_kp_steer
    )
)


File: configs/tutorial.yaml
pid_kp_roll=5.0
pid_ki_roll=0.0
pid_kd_roll=6.0
pid_kp_steer=1.1164231977942145


## Step 4. Leave the notebook

A notebook cell runs when you click it. Cells can run out of order, and an
old variable can still be in memory. That leftover is hidden state. A
controller needs a clock that fires every 0.02 s until you stop the process.
`sim/loop.py` owns that clock. `run_sim.py` is the single entry point, and
the numbers live in a YAML file that can be compared in version control.

Procedure:

1. Confirm `configs/tutorial.yaml` is saved.
2. Open a terminal in the `bike_mujoco_sim` folder. You may leave Jupyter
   running, or stop it with Ctrl+C in its terminal.
3. Activate the virtual environment so the prompt shows `(.venv)`.
4. Run one of the commands below.

Windows PowerShell:

```powershell
cd path\to\bike_mujoco_sim
.\.venv\Scripts\Activate.ps1
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode speed-schedule
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode remote
python run_sim.py --config configs/tutorial.yaml --model new_bike --headless --duration 6
```

macOS or Linux:

```bash
cd path/to/bike_mujoco_sim
source .venv/bin/activate
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode speed-schedule
python run_sim.py --config configs/tutorial.yaml --model new_bike --mode remote
python run_sim.py --config configs/tutorial.yaml --model new_bike --headless --duration 6
```

| Flag | Effect |
| --- | --- |
| `--config` | Which settings file to load |
| `--model new_bike` | Which bicycle model to load |
| `--mode speed-schedule` | Cruise only, no stick |
| `--mode remote` | 3D window plus joystick |
| `--headless` | No window; print a short report |


## Step 5. Check the saved gains inside Jupyter (optional)

This cell is a check, not a replacement for the terminal commands. It asks
whether the loaded YAML keeps the bicycle upright for 6 seconds.

If `configs/tutorial.yaml` is still the weak set (derivative gain below 8),
the cell loads `configs/new_bike.yaml` so the check can still run while you
are editing.

Expected result: a printed maximum |roll| under 20°.


In [4]:
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
# If the student has not saved tuned gains yet, fall back so Run All still works.
if cfg.pid_kd_roll < 8.0:
    print("tutorial.yaml still weak; verifying with new_bike.yaml for this check")
    cfg = load_config(new_bike_yaml)
else:
    print("verifying tutorial.yaml")
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=replace(cfg, duration=6.0))
history = run_headless(sim, duration=6.0)
summarize(history, model_path=sim.model_path, mode="deployment-check")
max_roll = float(np.max(np.abs(np.degrees(history["roll"]))))
print(f"max |roll| = {max_roll:.2f} deg")
if max_roll >= 20.0:
    raise RuntimeError("saved gains did not keep the bike upright")


tutorial.yaml still weak; verifying with new_bike.yaml for this check


model:      /home/lmbike2/bike-workshop/bike_mujoco_sim/models/new_bike/new_bike.xml
mode:       deployment-check
simulated:  6.00s
max |roll|: 6.56 deg
final speed:2.183 m/s
mean |roll|: 0.64 deg
max |roll| = 6.56 deg


## Discussion

### Why not leave the controller running in the notebook?

A cell runs when you click it, and cells can run out of order. A variable left over from an earlier cell can still change the next result. A controller needs a clock that fires every 0.02 s until you stop it. That clock lives in `sim/loop.py`, and `run_sim.py` is the program that starts it.

### Where do the gains go when the notebook kernel stops?

A number that exists only in a notebook variable disappears. The copy that remains is the one saved in `configs/tutorial.yaml`.

### What do I type in the terminal?

From the `bike_mujoco_sim` folder, `python run_sim.py --config configs/tutorial.yaml`. That starts the same controller as one process you can start and stop.


## Questions

Answer these before you run the next cell. Put your answers in the two strings at the top of the cell.

1. Which file do you edit to keep your gains?
2. Which flag runs the simulator with no window?
3. Predict a 4-second ride of the file as it is saved right now: fall, or hold under 45°?

The cell tells you whether each string matches this lesson, then rides `configs/tutorial.yaml` exactly as it is on disk. If that file is still the weak set, the result is a fall. Save the tuned gains and run the cell again to see a hold.


In [5]:
# Edit these two strings so they match your answers, then run.
file_you_edit = "configs/tutorial.yaml"
no_window_flag = "--headless"

file_ok = file_you_edit.replace("\\", "/").endswith("configs/tutorial.yaml")
flag_ok = no_window_flag.strip() == "--headless"
print("file:", "that is the working copy" if file_ok else "that is not the file this lesson asks you to edit")
print("flag:", "that runs with no window" if flag_ok else "that flag is not the headless run. Read the table in Step 4.")

tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
print(
    "saved kp={:.3f} ki={:.3f} kd={:.3f} kp_steer={:.3f}".format(
        cfg.pid_kp_roll, cfg.pid_ki_roll, cfg.pid_kd_roll, cfg.pid_kp_steer
    )
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=replace(cfg, duration=4.0))
history = run_headless(sim, duration=4.0)
roll_deg = np.degrees(np.asarray(history["roll"], dtype=float))
max_abs = float(np.max(np.abs(roll_deg)))
tip = np.where(np.abs(roll_deg) > 45.0)[0]
print(f"max |roll| = {max_abs:.1f} deg")
if len(tip) == 0:
    print("Result: held. The saved file kept roll under 45 deg.")
else:
    print("Result: fell. The saved file is still a weak set.")


file: that is the working copy
flag: that runs with no window
saved kp=5.000 ki=0.000 kd=6.000 kp_steer=1.116


max |roll| = 97.8 deg
Result: fell. The saved file is still a weak set.
